In [35]:
import pandas as pd
import json


In [36]:
# Leitura dos arquivos JSON e CSV na pasta dados-brutos.
df_clientes = pd.read_json("../dados-brutos/clientes.json")
df_pedidos = pd.read_json("../dados-brutos/pedidos.json")
df_produtos =pd.read_csv("../insumos/dados/produtos.csv")

In [37]:
# Não prcisa de tratamento, pois o arquivo já está limpo e pronta para ir para dados-tratados.
df_produtos

,produto_id,nome,categoria,preco_base
0,1,Alface crespa,folhas,3.5
1,2,Rucula,folhas,4.2
2,3,Tomate italiano,legumes,8.9
3,4,Cenoura,legumes,5.4
4,5,Batata inglesa,legumes,6.8
5,6,Banana prata,frutas,7.3
6,7,Maca fuji,frutas,9.9
7,8,Laranja pera,frutas,5.1
8,9,Morango,frutas,14.5
9,10,Abobora cabotia,legumes,4.7


CÓDIGO PROBLEMA:
- A1 Parte dos clientes tem e-mail ausente (valor nulo).
- A2 A sigla do estado aparece ora em maiúsculas
(RS), ora em minúsculas (rs).

O QUE SE ESPERA DO ETL:
- Manter o cliente, gravar o e-mail como nulo e contar quantos são.
- Padronizar para maiúsculas antes da carga.

In [38]:
# existe um padrão de criação dos emails de cada registros. email= nome+sobrenome(minusculo e separado por '.')+id+@+dominio.
df_clientes

,page,page_size,total_items,total_pages,items
0,1,200,1000,5,"{'id': 1, 'nome': 'Diego Fernandes', 'email': ..."
1,1,200,1000,5,"{'id': 2, 'nome': 'Zeca Ribeiro', 'email': 'ze..."
2,1,200,1000,5,"{'id': 3, 'nome': 'Diego Rocha', 'email': 'die..."
3,1,200,1000,5,"{'id': 4, 'nome': 'Marina Lima', 'email': 'mar..."
4,1,200,1000,5,"{'id': 5, 'nome': 'Ximena Fernandes', 'email':..."
...,...,...,...,...,...
195,1,200,1000,5,"{'id': 196, 'nome': 'Carla Souza', 'email': 'c..."
196,1,200,1000,5,"{'id': 197, 'nome': 'Ana Silva', 'email': None..."
197,1,200,1000,5,"{'id': 198, 'nome': 'Gabriela Costa', 'email':..."
198,1,200,1000,5,"{'id': 199, 'nome': 'Gabriela Souza', 'email':..."


In [ ]:
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           1000 non-null   int64 
 1   nome         1000 non-null   object
 2   email        919 non-null    object
 3   cidade       1000 non-null   object
 4   estado       1000 non-null   object
 5   cadastro_em  1000 non-null   object
dtypes: int64(1), object(5)
memory usage: 47.0+ KB


In [ ]:
# Guarda os valores originais para contabilizar as alterações.
estados_originais = df_clientes["estado"].copy()

# Remove espaços em branco e converte para maiúsculas.
df_clientes["estado"] = (
    df_clientes["estado"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conta quantos estados foram efetivamente padronizados.
estados_padronizados = (
    estados_originais.astype("string") != df_clientes["estado"]
).sum()
estados_padronizados

np.int64(151)

In [ ]:
df_clientes["estado"].value_counts(dropna=False)

estado
RS    314
SC    219
PR    191
SP    185
MG     91
Name: count, dtype: Int64

In [ ]:
df_clientes.isnull().sum()

id              0
nome            0
email          81
cidade          0
estado          0
cadastro_em     0
dtype: int64

In [ ]:
# Substituir valores nulos na coluna "email" por None.
df_clientes["email"] = df_clientes["email"].where(
    df_clientes["email"].notna(), None
)

In [ ]:
df_clientes['estado'].unique()  

In [ ]:
# Padronizando a coluna "estado" para letras maiúsculas.
df_clientes['estado'] = (
    df_clientes['estado']
    .str.strip()
    .str.upper()
)

In [ ]:
df_clientes['estado']

0      SP
1      MG
2      RS
3      PR
4      RS
       ..
995    PR
996    SC
997    SC
998    SC
999    SC
Name: estado, Length: 1000, dtype: object

CÓDIGO PROBLEMA:
- A3 A data do pedido vem em dois formatos: AAAAMM-DD e DD/MM/AAAA
- A4 O valor total do pedido vem ora como número,
ora como texto com vírgula decimal (por
exemplo, 125,84).
- A5 Alguns pedidos aparecem repetidos (mesmo id
e mesmo conteúdo) em páginas diferentes da
API.
- A6 Em parte dos pedidos, o valor total informado
não bate com a soma dos itens.


O QUE SE ESPERA DO ETL:
- Converter os dois formatos para o tipo de data do
banco (DATE no PostgreSQL; texto AAAA-MM-DD
no SQLite).
- Converter tudo para número com duas casas
decimais.
- Carregar cada pedido uma única vez; rodar o ETL
de novo não pode duplicar.
- Gravar os dois valores (valor_total e valor_itens),
marcar a divergência na coluna divergente e relatar
a quantidade.


In [7]:
# preicisaremos tratar a coluna itens pois ela está em formato de lista alinhada da tabela itens_pedidos.
df_pedidos

,id,cliente_id,data_pedido,status,valor_total,itens
0,1,303,2026-02-19,entregue,"125,84","[{'produto_id': 2, 'quantidade': 3, 'preco_uni..."
1,2,853,2026-08-21,entregue,94.34,"[{'produto_id': 6, 'quantidade': 4, 'preco_uni..."
2,3,358,2026-08-25,entregue,63.76,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
3,4,482,2026-05-21,entregue,"90,37","[{'produto_id': 2, 'quantidade': 4, 'preco_uni..."
4,5,728,2026-10-21,entregue,"9,52","[{'produto_id': 8, 'quantidade': 2, 'preco_uni..."
...,...,...,...,...,...,...
5095,4996,735,2026-04-03,entregue,31.5,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
5096,4997,909,2026-05-09,cancelado,68.32,"[{'produto_id': 11, 'quantidade': 6, 'preco_un..."
5097,4998,623,16/06/2026,entregue,"60,44","[{'produto_id': 4, 'quantidade': 5, 'preco_uni..."
5098,4999,390,07/02/2026,em_transporte,"9,92","[{'produto_id': 10, 'quantidade': 2, 'preco_un..."


In [21]:
# guardando a coluna original para contagem de datas convertidas.
datas_originais = df_pedidos["data_pedido"].copy()

# Converte a coluna data_pedido para datetime.
df_pedidos["data_pedido"] = pd.to_datetime(
    df_pedidos["data_pedido"],
    format="mixed",
    dayfirst=True
)

# Cria um avariavel para contar quantas datas foram convertidas.
datas_convertidas = (
    datas_originais.str.contains("/", na=False)
).sum()
datas_convertidas

np.int64(1054)

In [22]:
#padronizando a coluna valor_total para float
df_pedidos["valor_total"] = pd.to_numeric(
    df_pedidos["valor_total"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
).round(2)
df_pedidos

,id,cliente_id,data_pedido,status,valor_total,itens
0,1,303,2026-02-19,entregue,125.84,"[{'produto_id': 2, 'quantidade': 3, 'preco_uni..."
1,2,853,2026-08-21,entregue,94.34,"[{'produto_id': 6, 'quantidade': 4, 'preco_uni..."
2,3,358,2026-08-25,entregue,63.76,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
3,4,482,2026-05-21,entregue,90.37,"[{'produto_id': 2, 'quantidade': 4, 'preco_uni..."
4,5,728,2026-10-21,entregue,9.52,"[{'produto_id': 8, 'quantidade': 2, 'preco_uni..."
...,...,...,...,...,...,...
5095,4996,735,2026-04-03,entregue,31.50,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
5096,4997,909,2026-05-09,cancelado,68.32,"[{'produto_id': 11, 'quantidade': 6, 'preco_un..."
5097,4998,623,2026-06-16,entregue,60.44,"[{'produto_id': 4, 'quantidade': 5, 'preco_uni..."
5098,4999,390,2026-02-07,em_transporte,9.92,"[{'produto_id': 10, 'quantidade': 2, 'preco_un..."


In [ ]:
df_pedidos["valor_total"].dtype

dtype('float64')

In [23]:
# Removendo pedidos duplicados.
pedidos_unicos = df_pedidos.drop_duplicates(subset="id", keep="first")

# Criando DataFrame de itens a partir do JSON aninhado.
df_itens = pd.json_normalize(
    pedidos_unicos.to_dict("records"), record_path="itens", meta=["id"]
).rename(columns={"id": "pedido_id"})
# Criando sequência dos itens.
df_itens["linha"] = df_itens.groupby("pedido_id").cumcount() + 1

In [24]:
# Quantidade de pedidos antes da remoção.
total_pedidos = len(df_pedidos)

# Quantidade de pedidos após a remoção.
total_pedidos_unicos = len(pedidos_unicos)

# Quantidade de duplicados descartados.
repetidos_descartados = total_pedidos - total_pedidos_unicos

In [25]:
repetidos_descartados

100

In [26]:
# tabela itens_pedidos desalinhada com a tabela pedidos. pronta para ir para dados-tratados
df_itens

,produto_id,quantidade,preco_unitario,pedido_id,linha
0,2,3,4.30,1,1
1,9,5,15.86,1,2
2,7,4,9.66,1,3
3,6,4,7.34,2,1
4,7,3,10.40,2,2
...,...,...,...,...,...
12450,5,4,6.81,4997,4
12451,4,5,5.38,4998,1
12452,12,3,11.18,4998,2
12453,10,2,4.96,4999,1


In [14]:
pedidos_unicos

,id,cliente_id,data_pedido,status,valor_total,itens
0,1,303,2026-02-19,entregue,125.84,"[{'produto_id': 2, 'quantidade': 3, 'preco_uni..."
1,2,853,2026-08-21,entregue,94.34,"[{'produto_id': 6, 'quantidade': 4, 'preco_uni..."
2,3,358,2026-08-25,entregue,63.76,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
3,4,482,2026-05-21,entregue,90.37,"[{'produto_id': 2, 'quantidade': 4, 'preco_uni..."
4,5,728,2026-10-21,entregue,9.52,"[{'produto_id': 8, 'quantidade': 2, 'preco_uni..."
...,...,...,...,...,...,...
5095,4996,735,2026-04-03,entregue,31.50,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
5096,4997,909,2026-05-09,cancelado,68.32,"[{'produto_id': 11, 'quantidade': 6, 'preco_un..."
5097,4998,623,2026-06-16,entregue,60.44,"[{'produto_id': 4, 'quantidade': 5, 'preco_uni..."
5098,4999,390,2026-02-07,em_transporte,9.92,"[{'produto_id': 10, 'quantidade': 2, 'preco_un..."


In [15]:
# dataframe sem duplicidade
pedidos_unicos

,id,cliente_id,data_pedido,status,valor_total,itens
0,1,303,2026-02-19,entregue,125.84,"[{'produto_id': 2, 'quantidade': 3, 'preco_uni..."
1,2,853,2026-08-21,entregue,94.34,"[{'produto_id': 6, 'quantidade': 4, 'preco_uni..."
2,3,358,2026-08-25,entregue,63.76,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
3,4,482,2026-05-21,entregue,90.37,"[{'produto_id': 2, 'quantidade': 4, 'preco_uni..."
4,5,728,2026-10-21,entregue,9.52,"[{'produto_id': 8, 'quantidade': 2, 'preco_uni..."
...,...,...,...,...,...,...
5095,4996,735,2026-04-03,entregue,31.50,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
5096,4997,909,2026-05-09,cancelado,68.32,"[{'produto_id': 11, 'quantidade': 6, 'preco_un..."
5097,4998,623,2026-06-16,entregue,60.44,"[{'produto_id': 4, 'quantidade': 5, 'preco_uni..."
5098,4999,390,2026-02-07,em_transporte,9.92,"[{'produto_id': 10, 'quantidade': 2, 'preco_un..."


In [27]:
# Removendo a coluna "itens" do DataFrame de pedidos, pois os itens foram extraídos para o DataFrame df_itens.
pedidos_unicos = pedidos_unicos.drop(columns=["itens"])
pedidos_unicos.columns

Index(['id', 'cliente_id', 'data_pedido', 'status', 'valor_total'], dtype='object')

,produto_id,quantidade,preco_unitario,pedido_id,linha
0,2,3,4.30,1,1
1,9,5,15.86,1,2
2,7,4,9.66,1,3
3,6,4,7.34,2,1
4,7,3,10.40,2,2
...,...,...,...,...,...
12450,5,4,6.81,4997,4
12451,4,5,5.38,4998,1
12452,12,3,11.18,4998,2
12453,10,2,4.96,4999,1


In [29]:
# Calculando o total de cada item (quantidade * preço unitário).
# DataFrame auxiliar para consolidação dos valores.
df_itens_total = df_itens.copy()

# Calculando o valor total de cada item.
df_itens_total["total"] = (
    df_itens_total["quantidade"] *
    df_itens_total["preco_unitario"]
)
df_itens_total

,produto_id,quantidade,preco_unitario,pedido_id,linha,total
0,2,3,4.30,1,1,12.90
1,9,5,15.86,1,2,79.30
2,7,4,9.66,1,3,38.64
3,6,4,7.34,2,1,29.36
4,7,3,10.40,2,2,31.20
...,...,...,...,...,...,...
12450,5,4,6.81,4997,4,27.24
12451,4,5,5.38,4998,1,26.90
12452,12,3,11.18,4998,2,33.54
12453,10,2,4.96,4999,1,9.92


In [18]:
df_itens

,produto_id,quantidade,preco_unitario,pedido_id,linha,total
0,2,3,4.30,1,1,12.90
1,9,5,15.86,1,2,79.30
2,7,4,9.66,1,3,38.64
3,6,4,7.34,2,1,29.36
4,7,3,10.40,2,2,31.20
...,...,...,...,...,...,...
12450,5,4,6.81,4997,4,27.24
12451,4,5,5.38,4998,1,26.90
12452,12,3,11.18,4998,2,33.54
12453,10,2,4.96,4999,1,9.92


In [30]:
# somando o total de cada item para verificar se bate com o valor_total do pedido.
df_itens_total.groupby("pedido_id")["total"].sum()


pedido_id
1       130.84
2        94.34
3        63.76
4        90.37
5         9.52
         ...  
4996     31.50
4997     68.32
4998     60.44
4999      9.92
5000      8.88
Name: total, Length: 5000, dtype: float64

In [32]:
# Cria uma variavel que armazena o total de cada item agrupado por pedido_id.
valor_itens = (
        df_itens_total.groupby("pedido_id")["total"]
    .sum()
    .round(2)
)
valor_itens

pedido_id
1       130.84
2        94.34
3        63.76
4        90.37
5         9.52
         ...  
4996     31.50
4997     68.32
4998     60.44
4999      9.92
5000      8.88
Name: total, Length: 5000, dtype: float64

In [33]:
# Cria a coluna calculada valor_itens.
pedidos_unicos["valor_itens"] = (
    pedidos_unicos["id"].map(valor_itens)
)
pedidos_unicos

,id,cliente_id,data_pedido,status,valor_total,valor_itens
0,1,303,2026-02-19,entregue,125.84,130.84
1,2,853,2026-08-21,entregue,94.34,94.34
2,3,358,2026-08-25,entregue,63.76,63.76
3,4,482,2026-05-21,entregue,90.37,90.37
4,5,728,2026-10-21,entregue,9.52,9.52
...,...,...,...,...,...,...
5095,4996,735,2026-04-03,entregue,31.50,31.50
5096,4997,909,2026-05-09,cancelado,68.32,68.32
5097,4998,623,2026-06-16,entregue,60.44,60.44
5098,4999,390,2026-02-07,em_transporte,9.92,9.92


In [34]:
# Identificando divergência entre o valor total do pedido e a soma dos valores dos itens.
pedidos_unicos["divergente"] = (
    pedidos_unicos["valor_total"].round(2) !=
    pedidos_unicos["valor_itens"].round(2)
)
pedidos_unicos

,id,cliente_id,data_pedido,status,valor_total,valor_itens,divergente
0,1,303,2026-02-19,entregue,125.84,130.84,True
1,2,853,2026-08-21,entregue,94.34,94.34,False
2,3,358,2026-08-25,entregue,63.76,63.76,False
3,4,482,2026-05-21,entregue,90.37,90.37,False
4,5,728,2026-10-21,entregue,9.52,9.52,False
...,...,...,...,...,...,...,...
5095,4996,735,2026-04-03,entregue,31.50,31.50,False
5096,4997,909,2026-05-09,cancelado,68.32,68.32,False
5097,4998,623,2026-06-16,entregue,60.44,60.44,False
5098,4999,390,2026-02-07,em_transporte,9.92,9.92,False


In [ ]:
# quantidade de pedidos divergentes
pedidos_unicos["divergente"].sum()

np.int64(238)

In [ ]:
# Mostrando os pedidos divergentes.
pedidos_unicos[pedidos_unicos["divergente"]]

,id,cliente_id,data_pedido,status,valor_total,valor_itens,divergente
0,1,303,2026-02-19,entregue,125.84,130.84,True
50,51,23,2026-04-27,cancelado,26.58,23.58,True
70,71,962,2026-05-27,cancelado,144.35,134.35,True
80,81,198,2026-09-05,entregue,-1.84,3.16,True
105,106,973,02/05/2026,em_transporte,47.78,44.78,True
...,...,...,...,...,...,...,...
5018,4921,900,11/02/2026,entregue,40.10,37.10,True
5019,4922,624,2026-10-08,entregue,120.25,125.25,True
5079,4980,361,27/10/2026,em_transporte,52.50,57.50,True
5082,4983,228,2026-01-06,entregue,71.18,61.18,True


In [ ]:
##Escrevendo os DataFrames tratados em arquivos CSV na pasta dados-tratados.
#f_clientes.to_csv("../dados-tratados/clientes.csv", index=False, encoding="utf-8")
#f_produtos.to_csv("../dados-tratados/produtos.csv", index=False, encoding="utf-8")
#f_itens.to_csv("../dados-tratados/itens_pedido.csv", index=False, encoding="utf-8")
#edidos_unicos.to_csv("../dados-tratados/pedidos.csv", index=False, encoding="utf-8")